In [1]:
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query:str) -> str:
     """ Use this tool when you need to answer questions about current events or general knowledge. """
     from langchain_community.tools import DuckDuckGoSearchRun

     search = DuckDuckGoSearchRun()

     response = search.invoke(query)

     return response

In [4]:
tool_duckduckgo_search.invoke("what is the capital of france?")

'1 week ago - As the capital of France, Paris is the seat of France\'s national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The President of the French Republic resides at the Élysée Palace. 2 days ago - Its 18 integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi), with a total population estimated at over 69.1 million in 2026. Its capital, largest city and main cultural and economic centre is Paris. Metropolitan France was settled during the Iron ... March 10, 2026 - This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944. 4 days ago - France, a country of northwestern Europe, is historically and culturally among the most important countries in the Western world. It has also played a highly significant role in international affairs for centuries. Its capital is Paris, one of ... September 16, 2025 

In [5]:
@tool
def tool_wikipedia_search(query:str) -> str:
    """ Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response

In [6]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service. Loopt was acquired by Green Dot Corporation for $43.4 million. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. He is a billionaire with many investments including Reddit, Worldcoin, and Helion Energy.\nAltman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization\'s board of directors for not being "consistently candid". The move was met with significant backlash from employees and investors, resulting 

In [7]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics.

    Implementation notes:
    - Prefer the direct `arxiv` package (returns entries via `search.results()`).
    - If `arxiv` is not installed, fall back to the langchain_community wrapper/tool.
    - Return a formatted string (titles, urls, summaries) rather than printing.
    """

    try:
        import arxiv
    except Exception:
        # Fallback to langchain_community tool if `arxiv` package isn't available
        try:
            from langchain_community.tools import ArxivQueryRun
            from langchain_community.utilities import ArxivAPIWrapper

            arxiv_wrapper = ArxivAPIWrapper(
                top_k_results=3,
                doc_content_chars_max=2000,
            )
            arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
            if hasattr(arxiv_tool, "invoke"):
                return arxiv_tool.invoke(query)
            return arxiv_tool.run(query)
        except Exception as e:
            return f"arXiv wrapper error: {e}"

    # Use arxiv package directly using the Client API (arxiv v4+)
    try:
        search = arxiv.Search(query=query, max_results=3)
        client = arxiv.Client()

        entries = None
        try:
            entries = list(client.results(search))
        except Exception:
            entries = None

        # If client.results failed, try old-style fallbacks
        if entries is None:
            # Try Search.results() if present
            try:
                if hasattr(search, "results"):
                    entries = list(search.results())
            except Exception:
                entries = None

        if entries is None and hasattr(arxiv, "query"):
            try:
                entries = arxiv.query(query=query, max_results=3)
            except Exception:
                entries = None

        if not entries:
            return "No results found."

        parts = []
        for e in entries:
            if isinstance(e, dict):
                title = e.get("title", "")
                summary = e.get("summary", "").strip().replace("\n", " ")
                url = e.get("id", e.get("entry_id", ""))
            else:
                title = getattr(e, "title", "")
                summary = getattr(e, "summary", "").strip().replace("\n", " ")
                url = getattr(e, "entry_id", getattr(e, "id", ""))

            parts.append(f"Title: {title}\nURL: {url}\nSummary: {summary}\n")

        return "\n\n".join(parts)
    except Exception as exc:
        return f"arxiv package error: {exc}"

In [8]:
tool_arxiv_search.invoke("What are the latest research papers about deep learning for natural language processing?")

"Title: Modular Mechanistic Networks: On Bridging Mechanistic and Phenomenological Models with Deep Neural Networks in Natural Language Processing\nURL: http://arxiv.org/abs/1807.09844v2\nSummary: Natural language processing (NLP) can be done using either top-down (theory driven) and bottom-up (data driven) approaches, which we call mechanistic and phenomenological respectively. The approaches are frequently considered to stand in opposition to each other. Examining some recent approaches in deep learning we argue that deep neural networks incorporate both perspectives and, furthermore, that leveraging this aspect of deep learning may help in solving complex problems within language technology, such as modelling language and perception in the domain of spatial cognition.\n\n\nTitle: The Modern Mathematics of Deep Learning\nURL: http://arxiv.org/abs/2105.04026v2\nSummary: We describe the new field of mathematical analysis of deep learning. This field emerged around a list of research qu

In [9]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [10]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

## Integrating RAG Tool

In [11]:
if os.path.exists("../CH-4_Agent/chroma_db_semantic"):
    print("Yes exists")

Yes exists


In [12]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content

In [13]:
tool_rag.invoke("When was NovaSphere Organization founded?")

C:\Users\Hp\AppData\Local\Temp\ipykernel_20560\1495361009.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

## Bind Tools

In [14]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag
]

In [15]:
llm_bind = llm.bind_tools(toolkit)

In [16]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 374, 'total_tokens': 400, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1bfJWy1fZrz0uY7sdIwFKxMp8iB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d0-a505-7593-89bc-a5c80d93c873-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_EEMzui7xKSwmCkPgZBhQRB7y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 374, 'output_tokens': 26, 'total_tokens': 400, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## ReAct Agent

In [17]:
from langchain.agents import create_agent
my_agent = create_agent(llm_bind, toolkit)

In [18]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='2c882c00-538c-41f7-9e17-4e5dd4b9c1c9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 374, 'total_tokens': 464, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1bho2OS4YysTLAsIeSTXQxnbKY5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d0-ad8a-7950-9578-eebe6cfe47f2-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_PtnVxyldWHi77J5TT6p8in5e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [19]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

{'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='4af87131-8958-44a2-8e04-f775beb60af5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 368, 'total_tokens': 462, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1bmPIVxuWlSQzNVuQEzkZKiAm0l', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d0-c27a-7492-af36-742c8bece1c5-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_ByOLCWkSQmyDHQf2031mJVMv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [20]:
response =my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

In [21]:
response['messages']

[HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='e838971d-0f51-41b4-a650-ce952b457092'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 368, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1cZYlszaqn60LWZwCzFuKdsElU0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d1-7ded-71c0-803b-3b1e4880649a-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_DhmAtGRqgh63MuLV12sI97Rm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

## Memory

In [26]:
agent_memory = []

In [ ]:
# Option 1: Store the entire message history (including tool calls and responses)
def invoke_agent_with_memory(query: str):

    reponse = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query: {query}. You can refer to previous interactions : {str(agent_memory)}"}]}
    )
    agent_memory.append({"query": query, "response": response})
    return reponse

In [ ]:
# Option 2: Store only the relevant information (e.g., tool calls and responses) rather than the entire message history
def invoke_agent_with_memory(query: str):

    reponse = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query: {query}. You can refer to previous interactions : {str(agent_memory)}"}]}
    )
    agent_memory.append(response['messages'])
    return reponse

In [ ]:
# Option 3:Store only the relevant information (e.g., tool calls and responses) rather than the entire message history, but use extend instead of append to flatten the list
def invoke_agent_with_memory(query: str):

    reponse = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query: {query}. You can refer to previous interactions : {str(agent_memory)}"}]}
    )
    agent_memory.extend(response['messages'])
    return reponse

In [35]:
invoke_agent_with_memory("When was NovaSphere Organization founded?")

{'messages': [HumanMessage(content="Answer this query: When was NovaSphere Organization founded?. You can refer to previous interactions : [{'query': 'When was NovaSphere Organization founded?', 'response': {'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='e838971d-0f51-41b4-a650-ce952b457092'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 368, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1cZYlszaqn60LWZwCzFuKdsElU0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d1-7ded-71c0-803b-3

In [36]:
invoke_agent_with_memory("Could you please summarize the previous interactions?")

{'messages': [HumanMessage(content="Answer this query: Could you please summarize the previous interactions?. You can refer to previous interactions : [{'query': 'When was NovaSphere Organization founded?', 'response': {'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='e838971d-0f51-41b4-a650-ce952b457092'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 368, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1cZYlszaqn60LWZwCzFuKdsElU0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d1-7ded

In [30]:
agent_memory

[{'query': 'When was NovaSphere Organization founded?',
  'response': {'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='e838971d-0f51-41b4-a650-ce952b457092'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 368, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1cZYlszaqn60LWZwCzFuKdsElU0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d1-7ded-71c0-803b-3b1e4880649a-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_DhmAtGRq

In [ ]:
while True:
    query = input("Ask me anything: ")
    response = invoke_agent_with_memory(query)
    print(response)
    flag = input("Do you want to continue? (yes/no): ")
    if flag.lower() == "no":
        break

{'messages': [HumanMessage(content="Answer this query: Hi my name is Gopi. You can refer to previous interactions : [{'query': 'When was NovaSphere Organization founded?', 'response': {'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='e838971d-0f51-41b4-a650-ce952b457092'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 368, 'total_tokens': 398, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj1cZYlszaqn60LWZwCzFuKdsElU0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e59d1-7ded-71c0-803b-3b1e4880649a-0', tool_ca

In [ ]:
agent_memory